### Confirm Single Molecule Tracks 2

In this cell, each detected transition is filtered by step size and SNR: step_size > 0 and snr >= MIN_SNR, and the class is based on n_valid_steps

In [ ]:
# -------------------------------------------------------------
# MULTI-STEP PHOTOBLEACHING ANALYSIS
# -------------------------------------------------------------
INTENSITY_COL = 'mass'
MIN_TRACK_LENGTH = 15
MIN_EDGE_FRAMES = 5
MIN_SNR = 3.0
MAX_FINAL_RATIO = 0.4
MAX_STEPS = 4
BIC_THRESHOLD = 10.0

def _segment_rss(y):
    n = len(y)
    if n <= 1:
        return 0.0
    s = y.sum()
    ss = np.dot(y, y)
    return float(ss - (s * s) / n)

def _best_split_local(y, min_edge=5):
    n = len(y)
    if n < 2 * min_edge:
        return None

    csum = np.cumsum(y)
    csum2 = np.cumsum(y * y)
    k = np.arange(min_edge, n - min_edge + 1)

    left_sum = csum[min_edge - 1 : n - min_edge]
    right_sum = csum[-1] - left_sum
    left_n = k
    right_n = n - k

    left_rss = csum2[min_edge - 1 : n - min_edge] - (left_sum * left_sum) / left_n
    right_rss = (csum2[-1] - csum2[min_edge - 1 : n - min_edge]) - (right_sum * right_sum) / right_n
    total_rss = left_rss + right_rss

    j = int(np.argmin(total_rss))
    return int(k[j]), float(total_rss[j])

def fit_multi_step(frames, intensities, min_edge=5, max_steps=4, bic_threshold=10.0):
    y = np.asarray(intensities, dtype=float)
    n = len(y)
    if n < 2 * min_edge:
        return None

    segments = [(0, n)]
    boundaries = []

    for _ in range(max_steps - 1):
        best_gain = 0.0
        best_choice = None

        rss_before = sum(_segment_rss(y[s:e]) for s, e in segments)
        p_before = len(segments)
        bic_before = n * np.log(max(rss_before / n, 1e-12)) + p_before * np.log(n)

        for seg_idx, (s, e) in enumerate(segments):
            seg_len = e - s
            if seg_len < 2 * min_edge:
                continue

            local = _best_split_local(y[s:e], min_edge=min_edge)
            if local is None:
                continue

            k_local, split_rss = local
            k_global = s + k_local

            rss_candidate = 0.0
            for i, (ss, ee) in enumerate(segments):
                if i == seg_idx:
                    rss_candidate += split_rss
                else:
                    rss_candidate += _segment_rss(y[ss:ee])

            p_after = p_before + 1
            bic_after = n * np.log(max(rss_candidate / n, 1e-12)) + p_after * np.log(n)
            gain = bic_before - bic_after

            if gain > best_gain:
                best_gain = gain
                best_choice = (seg_idx, k_global)

        if best_choice is None or best_gain < bic_threshold:
            break

        seg_idx, k_global = best_choice
        s, e = segments[seg_idx]
        segments = segments[:seg_idx] + [(s, k_global), (k_global, e)] + segments[seg_idx + 1 :]
        boundaries.append(k_global)

    boundaries = sorted(set(boundaries))
    if not boundaries:
        return None

    edges = [0] + boundaries + [n]
    state_means = []
    state_stds = []
    state_ranges = []
    for i in range(len(edges) - 1):
        s, e = edges[i], edges[i + 1]
        yy = y[s:e]
        if len(yy) == 0:
            continue
        state_means.append(float(np.mean(yy)))
        state_stds.append(float(np.std(yy)) if len(yy) > 1 else 1e-6)
        state_ranges.append((s, e))

    valid_steps = []
    all_steps = []
    for i in range(len(state_means) - 1):
        step_size = state_means[i] - state_means[i + 1]
        noise_next = max(state_stds[i + 1], 1e-6)
        snr = step_size / noise_next
        step_frame = int(frames[edges[i + 1]]) if edges[i + 1] < len(frames) else int(frames[-1])
        step_info = {
            'step_idx': int(edges[i + 1]),
            'step_frame': step_frame,
            'step_size': float(step_size),
            'snr': float(snr),
        }
        all_steps.append(step_info)
        if step_size > 0 and snr >= MIN_SNR:
            valid_steps.append(step_info)

    # Keep only tracks with at least one valid bleach step for reporting
    if len(valid_steps) < 1:
        return None

    initial_mean = state_means[0]
    final_mean = state_means[-1]
    final_ratio = final_mean / initial_mean if initial_mean != 0 else np.inf

    if final_ratio >= MAX_FINAL_RATIO:
        return None

    return {
        'boundaries': boundaries,
        'states': state_ranges,
        'state_means': state_means,
        'all_steps': all_steps,
        'valid_steps': valid_steps,
        'n_valid_steps': len(valid_steps),
        'initial_mean': float(initial_mean),
        'final_mean': float(final_mean),
        'final_ratio': float(final_ratio),
    }

step_results = []
all_tracks = df.groupby('UID')

for uid, track_df in all_tracks:
    track_df = track_df.sort_values('Frame').dropna(subset=[INTENSITY_COL])
    if len(track_df) < MIN_TRACK_LENGTH:
        continue

    frames = track_df['Frame'].values
    intensities = track_df[INTENSITY_COL].values

    fit = fit_multi_step(
        frames,
        intensities,
        min_edge=MIN_EDGE_FRAMES,
        max_steps=MAX_STEPS,
        bic_threshold=BIC_THRESHOLD,
    )
    if fit is None:
        continue

    step_results.append({
        'UID': uid,
        'track_length': len(track_df),
        'n_valid_steps': fit['n_valid_steps'],
        'step_frames': [s['step_frame'] for s in fit['valid_steps']],
        'step_sizes': [round(s['step_size'], 2) for s in fit['valid_steps']],
        'step_snrs': [round(s['snr'], 2) for s in fit['valid_steps']],
        'initial_mean': fit['initial_mean'],
        'final_mean': fit['final_mean'],
        'final_ratio': fit['final_ratio'],
    })

step_df = pd.DataFrame(step_results)
single_step_df = pd.DataFrame()
multi_df = pd.DataFrame()
stepwise_uids = []
single_step_uids = []
multistep_uids = []

print("\n" + "=" * 62)
print("STEP-WISE PHOTOBLEACHING ANALYSIS SUMMARY")
print("=" * 62)
if step_df.empty:
    print("No trajectories with >=1 valid bleaching step found with current thresholds.")
else:
    n_total_tracks = df['UID'].nunique()
    n_stepwise = len(step_df)

    single_step_df = step_df[step_df['n_valid_steps'] == 1].copy()
    multi_df = step_df[step_df['n_valid_steps'] >= 2].copy()
    n_single = len(single_step_df)
    n_multistep = len(multi_df)

    print(f"Total trajectories in dataset:                         {n_total_tracks}")
    print(f"Trajectories with >=1 valid bleaching step:            {n_stepwise}")
    print(f"Tracks with exactly 1 step:                            {n_single}")
    print(f"Tracks with >=2 steps:                                 {n_multistep}")

    step_counts = step_df['n_valid_steps'].value_counts().sort_index()
    print("\nStep-count distribution (valid steps per trajectory):")
    for steps, count in step_counts.items():
        pct = 100 * count / n_stepwise
        print(f"  {int(steps)} steps: {count} tracks ({pct:.1f}%)")

    print("\nTop 25 step-wise trajectories (sorted by step count, then final ratio):")
    display_cols = ['UID', 'n_valid_steps', 'step_frames', 'step_sizes', 'step_snrs', 'final_ratio', 'track_length']
    step_df = step_df.sort_values(['n_valid_steps', 'final_ratio'], ascending=[False, True]).reset_index(drop=True)
    display(step_df[display_cols].head(25))

    stepwise_uids = step_df['UID'].tolist()
    single_step_uids = single_step_df.sort_values(['n_valid_steps', 'final_ratio'], ascending=[False, True])['UID'].tolist()
    multistep_uids = multi_df.sort_values(['n_valid_steps', 'final_ratio'], ascending=[False, True])['UID'].tolist()

    print(f"\nStored {len(stepwise_uids)} IDs in variable: stepwise_uids")
    print(f"Stored {len(single_step_uids)} IDs in variable: single_step_uids")
    print(f"Stored {len(multistep_uids)} IDs in variable: multistep_uids")
print("=" * 62)

# Optional compact chart of step-count proportions
if not step_df.empty:
    hv.renderer('bokeh').theme = Theme(filename='util/bokeh-theme-light.yaml')
    counts = step_df['n_valid_steps'].value_counts().sort_index()
    proportions = (counts / counts.sum() * 100).round(2)
    proportion_df = pd.DataFrame({'steps': counts.index.astype(str), 'percent': proportions.values})
    proportion_bars = hv.Bars(proportion_df, kdims='steps', vdims='percent').opts(
        title='Step-wise Trajectory Proportions',
        xlabel='Number of Valid Bleaching Steps',
        ylabel='Percent of Step-wise Tracks (%)',
        color='#4E79A7',
        width=650,
        height=350,
    ).opts(laf.HV_BOKEH_BASIC).opts(hooks=[laf.bokeh_add_topright_linear_axes])

proportion_bars


STEP-WISE PHOTOBLEACHING ANALYSIS SUMMARY
Total trajectories in dataset:                         7747
Trajectories with >=1 valid bleaching step:            219
Tracks with exactly 1 step:                            166
Tracks with >=2 steps:                                 53

Step-count distribution (valid steps per trajectory):
  1 steps: 166 tracks (75.8%)
  2 steps: 52 tracks (23.7%)
  3 steps: 1 tracks (0.5%)

Top 25 step-wise trajectories (sorted by step count, then final ratio):


,UID,n_valid_steps,step_frames,step_sizes,step_snrs,final_ratio,track_length
0,4mW20ms 25 B 5-30 min00043-1898,3,"[22, 38, 70]","[488.23, 398.28, 443.63]","[3.89, 3.63, 4.83]",0.152785,204
1,4mW20ms 25 B 5-30 min00052-4051,2,"[14, 41]","[246.9, 227.46]","[3.09, 6.78]",0.172797,43
2,4mW20ms 25 B 5-30 min00046-830,2,"[39, 66]","[318.57, 195.08]","[3.14, 4.55]",0.181071,76
3,4mW20ms 25 B 5-30 min00046-275,2,"[10, 23]","[494.55, 365.91]","[5.52, 4.76]",0.191406,70
4,4mW20ms 25 B 5-30 min00045-5772,2,"[551, 585]","[325.0, 238.94]","[10.43, 15.22]",0.203591,187
5,4mW20ms 25 B 5-30 min00043-1677,2,"[11, 36]","[460.24, 152.59]","[5.04, 4.45]",0.212623,58
6,4mW20ms 25 B 5-30 min00046-501,2,"[11, 49]","[363.68, 205.8]","[5.67, 4.52]",0.215694,54
7,4mW20ms 25 B 5-30 min00043-1551,2,"[9, 38]","[413.01, 143.99]","[8.04, 4.54]",0.226638,76
8,4mW20ms 25 B 5-30 min00040-1325,2,"[33, 40]","[189.47, 154.52]","[7.16, 7.91]",0.232009,98
9,4mW20ms 25 B 5-30 min00043-1702,2,"[23, 56]","[247.26, 123.09]","[4.4, 5.2]",0.240009,57



Stored 219 IDs in variable: stepwise_uids
Stored 166 IDs in variable: single_step_uids
Stored 53 IDs in variable: multistep_uids


:Bars   [steps]   (percent)

In [ ]:
# Export step-wise and multistep trajectories for downstream checks
if 'step_df' in globals() and not step_df.empty:
    ordered_step_df = step_df.sort_values(['n_valid_steps', 'final_ratio'], ascending=[False, True]).reset_index(drop=True)

    stepwise_path = data_wd / 'stepwise_trajectories.csv'
    ordered_step_df.to_csv(stepwise_path, index=False)
    print(f"Saved step-wise table (>=1 step): {stepwise_path}")

    ordered_multistep_df = ordered_step_df[ordered_step_df['n_valid_steps'] >= 2].copy()
    multistep_path = data_wd / 'multistep_trajectories.csv'
    ordered_multistep_df.to_csv(multistep_path, index=False)
    print(f"Saved multistep table (>=2 steps): {multistep_path}")

    one_step_count = int((ordered_step_df['n_valid_steps'] == 1).sum())
    multi_step_count = int((ordered_step_df['n_valid_steps'] >= 2).sum())
    print(f"\nCount summary: 1-step={one_step_count}, >=2-steps={multi_step_count}")
else:
    print('No step-wise trajectories found.')